In [38]:
!pip install gymnasium stable-baselines3[extra] boto3

In [47]:
# Cell 2
import gymnasium as gym
from gymnasium import spaces
import numpy as np

class SlipperyKitchenEnv(gym.Env):
    metadata = {"render_modes": ["human"]}

    def __init__(self, grid_size=5, slip_probability=0.1, render_mode=None):
        super(SlipperyKitchenEnv, self).__init__()
        self.grid_size = grid_size
        self.slip_probability = slip_probability
        self.render_mode = render_mode

        self.action_space = spaces.Discrete(4)
        self.observation_space = spaces.Box(
            low=0, high=255,
            shape=(3, self.grid_size, self.grid_size),
            dtype=np.uint8
        )
        self.max_steps = 200 # Max steps before giving up
        self.current_step = 0

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.current_step = 0 # Reset the clock
        self.agent_pos = np.array([self.np_random.integers(0, self.grid_size),
                                   self.np_random.integers(0, self.grid_size)])
        self.dish_pos = np.array([self.np_random.integers(0, self.grid_size),
                                  self.np_random.integers(0, self.grid_size)])
        self.sink_pos = np.array([self.np_random.integers(0, self.grid_size),
                                  self.np_random.integers(0, self.grid_size)])
        self.has_dish = False

        if self.render_mode == "human":
            self.render()

        return self._get_obs(), {}

    def _get_manhattan_distance(self, pos1, pos2):
        return np.abs(pos1[0] - pos2[0]) + np.abs(pos1[1] - pos2[1])

    def step(self, action):
        self.current_step += 1

        # --- 1. PRE-MOVEMENT DISTANCE ---
        # Determine current target
        target_pos = self.sink_pos if self.has_dish else self.dish_pos
        # Calculate Manhattan distance: |x1 - x2| + |y1 - y2|
        old_distance = abs(self.agent_pos[0] - target_pos[0]) + abs(self.agent_pos[1] - target_pos[1])

        # --- 2. MOVEMENT & SLIP LOGIC ---
        if self.np_random.random() < self.slip_probability:
            possible_slips = [a for a in range(4) if a != action]
            action = self.np_random.choice(possible_slips)

        if action == 0:   # Up
            self.agent_pos[1] = min(self.grid_size - 1, self.agent_pos[1] + 1)
        elif action == 1: # Right
            self.agent_pos[0] = min(self.grid_size - 1, self.agent_pos[0] + 1)
        elif action == 2: # Down
            self.agent_pos[1] = max(0, self.agent_pos[1] - 1)
        elif action == 3: # Left
            self.agent_pos[0] = max(0, self.agent_pos[0] - 1)

        # --- 3. REWARDS & TASK LOGIC ---
        reward = -0.01
        terminated = False
        just_picked_up = False # Flag to prevent broken shaping math

        if not self.has_dish and np.array_equal(self.agent_pos, self.dish_pos):
            self.has_dish = True
            reward += 0.5
            self.dish_pos = np.array([-1, -1])
            just_picked_up = True

        if self.has_dish and np.array_equal(self.agent_pos, self.sink_pos):
            reward += 1.0
            terminated = True

        # --- 4. APPLY DENSE REWARD SHAPING ---
        # Only shape if the target didn't just change, and the game isn't over
        if not just_picked_up and not terminated:
            new_distance = abs(self.agent_pos[0] - target_pos[0]) + abs(self.agent_pos[1] - target_pos[1])
            distance_delta = old_distance - new_distance

            # Positive delta = moved closer. Negative delta = moved further.
            shaping_reward = distance_delta * 0.02
            reward += shaping_reward

        if self.render_mode == "human":
            self.render()

        truncated = self.current_step >= self.max_steps

        return self._get_obs(), float(reward), terminated, truncated, {}

    def _get_obs(self):
        obs = np.zeros((3, self.grid_size, self.grid_size), dtype=np.uint8)
        obs[0, self.agent_pos[1], self.agent_pos[0]] = 255
        if not self.has_dish:
            obs[1, self.dish_pos[1], self.dish_pos[0]] = 255
        obs[2, self.sink_pos[1], self.sink_pos[0]] = 255
        return obs

    def render(self):
        grid = np.full((self.grid_size, self.grid_size), '.')
        grid[self.sink_pos[1], self.sink_pos[0]] = 'S'
        if not self.has_dish:
            grid[self.dish_pos[1], self.dish_pos[0]] = 'D'
        grid[self.agent_pos[1], self.agent_pos[0]] = 'A'

        print("\n".join(" ".join(row) for row in grid))
        print("-" * 10)

In [48]:
# Cell 3
import torch as th
import torch.nn as nn
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor

class TinyCNN(BaseFeaturesExtractor):
    def __init__(self, observation_space: gym.spaces.Box, features_dim: int = 128):
        super().__init__(observation_space, features_dim)
        n_input_channels = observation_space.shape[0]

        self.cnn = nn.Sequential(
            nn.Conv2d(n_input_channels, 16, kernel_size=2, stride=1, padding=0),
            nn.ReLU(),
            nn.Conv2d(16, 32, kernel_size=2, stride=1, padding=0),
            nn.ReLU(),
            nn.Flatten(),
        )

        with th.no_grad():
            sample_obs = th.as_tensor(observation_space.sample()[None]).float()
            n_flatten = self.cnn(sample_obs).shape[1]

        self.linear = nn.Sequential(
            nn.Linear(n_flatten, features_dim),
            nn.ReLU()
        )

    def forward(self, observations: th.Tensor) -> th.Tensor:
        return self.linear(self.cnn(observations))

In [49]:
# Cell 4
from stable_baselines3 import PPO
from stable_baselines3.common.env_checker import check_env

# 1. Check environment
env = SlipperyKitchenEnv(grid_size=8)
check_env(env)
print("Environment check passed!")

# 2. Setup Model
policy_kwargs = dict(
    features_extractor_class=TinyCNN,
    features_extractor_kwargs=dict(features_dim=128),
)

model = PPO(
    "CnnPolicy",
    env,
    policy_kwargs=policy_kwargs,
    verbose=1,
    learning_rate=0.0005,
    n_steps=1024,
    batch_size=64
)

# 3. Train
print("Starting training...")
model.learn(total_timesteps=300000)

# 4. Save locally
model.save("ppo_slippery_kitchen")
print("Model saved!")

Streaming output truncated to the last 5000 lines.
-----------------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 163         |
|    ep_rew_mean          | -0.933      |
| time/                   |             |
|    fps                  | 379         |
|    iterations           | 56          |
|    time_elapsed         | 150         |
|    total_timesteps      | 57344       |
| train/                  |             |
|    approx_kl            | 0.057122543 |
|    clip_fraction        | 0.327       |
|    clip_range           | 0.2         |
|    entropy_loss         | -0.795      |
|    explained_variance   | 0.106       |
|    learning_rate        | 0.0005      |
|    loss                 | -0.0579     |
|    n_updates            | 550         |
|    policy_gradient_loss | -0.0459     |
|    value_loss           | 0.0202      |
-----------------------------------------
-------------------------

In [50]:
# Cell 6
import time

# Create a new environment for rendering
eval_env = SlipperyKitchenEnv(grid_size=8, render_mode="human") # Set grid_size to 8
obs, info = eval_env.reset()

print("Initial State:")
eval_env.render()

for i in range(50): # Run for 20 steps
    # Predict the best action
    action, _states = model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = eval_env.step(action)

    print(f"Step {i+1} | Reward: {reward}")
    time.sleep(0.5) # Pause to make it readable

    if terminated:
        print("Success! The agent completed the task.")
        break

. . . . . . . .
. . . . . . . .
. . A . . . . .
. . . D . . . .
. . . . . . . S
. . . . . . . .
. . . . . . . .
. . . . . . . .
----------
Initial State:
. . . . . . . .
. . . . . . . .
. . A . . . . .
. . . D . . . .
. . . . . . . S
. . . . . . . .
. . . . . . . .
. . . . . . . .
----------
. . . . . . . .
. . . . . . . .
. . . A . . . .
. . . D . . . .
. . . . . . . S
. . . . . . . .
. . . . . . . .
. . . . . . . .
----------
Step 1 | Reward: 0.01
. . . . . . . .
. . . . . . . .
. . . . . . . .
. . . A . . . .
. . . . . . . S
. . . . . . . .
. . . . . . . .
. . . . . . . .
----------
Step 2 | Reward: 0.49
. . . . . . . .
. . . . . . . .
. . . . . . . .
. . . . . . . .
. . . A . . . S
. . . . . . . .
. . . . . . . .
. . . . . . . .
----------
Step 3 | Reward: 0.01
. . . . . . . .
. . . . . . . .
. . . . . . . .
. . . . . . . .
. . . . A . . S
. . . . . . . .
. . . . . . . .
. . . . . . . .
----------
Step 4 | Reward: 0.01
. . . . . . . .
. . . . . . . .
. . . . . . . .
. . . . . . . .

In [52]:
import numpy as np
import imageio
from PIL import Image, ImageDraw
from IPython.display import Video

# Action mapping for readable labels
ACTION_NAMES = {0: "Up", 1: "Right", 2: "Down", 3: "Left"}

def create_fully_labeled_frame(env, step, reward, action_name="Start"):
    cell_size = 80  # Slightly bigger cells to fit text comfortably
    grid_size = env.unwrapped.grid_size
    width = grid_size * cell_size
    header_height = 60
    height = width + header_height

    # Create canvas
    img = Image.new('RGB', (width, height), color=(255, 255, 255))
    draw = ImageDraw.Draw(img)

    # --- 1. Draw Header ---
    # Using fixed-width formatting {: <6} for action name to keep alignment steady
    text_str = f"Step: {step:<2} | Action: {action_name: <6} | Reward: {reward: .2f}"
    draw.text((15, 20), text_str, fill=(0, 0, 0))
    draw.line([(0, header_height - 1), (width, header_height - 1)], fill=(0, 0, 0), width=3)

    # --- 2. Draw Grid and Labels ---
    for y in range(grid_size):
        for x in range(grid_size):
            # Calculate cell boundaries
            top_left_x = x * cell_size
            top_left_y = header_height + (y * cell_size)
            bottom_right_x = top_left_x + cell_size
            bottom_right_y = top_left_y + cell_size

            # Default background (empty floor)
            fill_color = (235, 235, 235)
            cell_label = None
            label_color = (0,0,0)

            # Check what objects are in this cell
            pos = np.array([x, y])

            # Check Sink first (so Agent can stand ON it)
            if np.array_equal(pos, env.unwrapped.sink_pos):
                fill_color = (100, 150, 255) # Blue
                cell_label = "S"

            # Check Dish
            if not env.unwrapped.has_dish and np.array_equal(pos, env.unwrapped.dish_pos):
                fill_color = (100, 255, 100) # Green
                cell_label = "D"

            # Check Agent last (draws on top)
            if np.array_equal(pos, env.unwrapped.agent_pos):
                fill_color = (255, 100, 100) # Red
                cell_label = "A"
                # If agent has dish, indicate that
                if env.unwrapped.has_dish:
                    cell_label = "A+D"

            # Draw the colored square
            draw.rectangle(
                [(top_left_x, top_left_y), (bottom_right_x, bottom_right_y)],
                fill=fill_color, outline=(50, 50, 50), width=1
            )

            # Draw the text label centered in the square
            if cell_label:
                # Basic centering estimate for default PIL font
                text_x = top_left_x + (cell_size // 2) - (len(cell_label) * 3)
                text_y = top_left_y + (cell_size // 2) - 5
                draw.text((text_x, text_y), cell_label, fill=label_color)

    return np.array(img)

# --- Run Evaluation Loop ---
eval_env = SlipperyKitchenEnv(grid_size=8)
obs, info = eval_env.reset()
frames = []

print("Generating fully labeled video frames...")

# Initial State Frame
frames.append(create_fully_labeled_frame(eval_env, step=0, reward=0.0, action_name="Start"))

for step in range(100):
    # Get action from model
    action_idx, _states = model.predict(obs, deterministic=True)
    action_name = ACTION_NAMES[action_idx.item()] # Fixed: Convert numpy array to scalar

    # Step environment
    obs, reward, terminated, truncated, info = eval_env.step(action_idx)

    # Create frame with the action that just happened and resulting reward
    frames.append(create_fully_labeled_frame(eval_env, step=step+1, reward=reward, action_name=action_name))

    if terminated:
        # Hold final frame
        final_frame = create_fully_labeled_frame(eval_env, step=step+1, reward=reward, action_name="DONE")
        for _ in range(6): frames.append(final_frame)
        print(f"FINISHED. Task complete in {step+1} steps.")
        break

# Save and display
video_filename = "fully_labeled_demo.mp4"
# Using 2 FPS so you have time to read the action label before it changes
imageio.mimsave(video_filename, frames, fps=2)
print(f"Video saved to {video_filename}")

Video(video_filename, embed=True)

Generating fully labeled video frames...
FINISHED. Task complete in 12 steps.
Video saved to fully_labeled_demo.mp4
